In [ ]:
%%capture
!pip install llama-index llama-index-embeddings-openai qdrant-client llama-index-vector-stores-qdrant llama-index llama-index-llms-openai llama-index-vector-stores-faiss faiss-cpu

In [ ]:
import os
import pickle
import faiss
import numpy as np
from getpass import getpass
import nest_asyncio
from dotenv import load_dotenv
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.settings import Settings
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.extractors import TitleExtractor, SummaryExtractor
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.vector_stores.faiss import FaissVectorStore
from naive_rag.helpers.IngestionCacheManager import SmartIngestionCache
from llama_index.core.settings import Settings

# Define file paths for caching
CACHE_FILE = "./pipeline_storage/nodes_cache.pkl"
INDEX_FILE = "./pipeline_storage/faiss_index.index"
PERSIST_DIR = "./pipeline_storage"

nest_asyncio.apply()

load_dotenv()

In [ ]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY') or getpass("Enter your OpenAI API key: ")
TAVILY_API_KEY=os.environ.get('TAVILY_API_KEY') or getpass("Enter your TAVILY API key: ")
ANTHROPIC_API_KEY=os.environ.get('ANTHROPIC_API_KEY') or getpass("Enter your ANTHROPIC API key: ")

In [ ]:
Settings.llm = OpenAI(model="gpt-4o-mini-2024-07-18", api_key=OPENAI_API_KEY)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [ ]:
# ----------------------- Helper Functions -----------------------

def save_nodes(nodes, cache_file=CACHE_FILE):
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)
    with open(cache_file, "wb") as f:
        pickle.dump(nodes, f)
    print("Nodes saved locally.")

def load_nodes(cache_file=CACHE_FILE):
    if os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            nodes = pickle.load(f)
        print("Loaded nodes from cache.")
        return nodes
    return None

def save_faiss_index(index, index_file=INDEX_FILE):
    os.makedirs(os.path.dirname(index_file), exist_ok=True)
    faiss.write_index(index, index_file)
    print("FAISS index saved locally.")

def load_faiss_index(d=1536, index_file=INDEX_FILE):
    if os.path.exists(index_file):
        index = faiss.read_index(index_file)
        print("Loaded FAISS index from cache.")
        return index
    return None

def build_pipeline():
    """Builds the ingestion pipeline for processing documents into nodes."""
    transformations = [
        SentenceSplitter(chunk_size=500, chunk_overlap=40),
        TitleExtractor(
            llm=Settings.llm,
            metadata_mode=MetadataMode.EMBED,
            num_workers=6
        ),
        SummaryExtractor(
            llm=Settings.llm,
            metadata_mode=MetadataMode.EMBED,
            num_workers=6
        ),
        Settings.embed_model
    ]
    return IngestionPipeline(transformations=transformations)

def load_documents(path):
    """Loads documents from the specified directory."""
    reader = SimpleDirectoryReader(path)
    return reader.load_data()

def create_faiss_index(nodes, d=1536, nlist=32, m=16, bits=4):
    """
    Creates and trains a FAISS index using embeddings extracted from the nodes.
    It first attempts to load a cached index from disk.
    """
    # Compute embeddings from nodes
    embeddings = np.stack([node.get_embedding() for node in nodes]).astype("float32")

    # Try to load an existing FAISS index
    faiss_index = load_faiss_index(d)
    if faiss_index is None:
        # Create a new FAISS index
        faiss_index = faiss.index_factory(d, f"IVF{nlist},PQ{m}x{bits}")
        if not faiss_index.is_trained:
            print("Training the FAISS index...")
            faiss_index.train(embeddings)
            print("Index trained.")
        save_faiss_index(faiss_index)
    return faiss_index



In [ ]:
# ----------------------- Main Pipeline Execution -----------------------

# 1. Load your documents from disk
documents = load_documents(r"C:\Users\anteb\PycharmProjects\JupyterProject\naive_rag\data\ai_articles")

# 2. Load cached nodes if available; otherwise, build the pipeline and process documents
nodes = load_nodes()
base_pipeline = build_pipeline()
if nodes is None:
    print("No cached nodes found; processing documents to extract embeddings...")
    nodes = base_pipeline.run(documents=documents)
    save_nodes(nodes)

# 3. Create or load a FAISS index based on the nodes embeddings
faiss_index = create_faiss_index(nodes)
vector_store = FaissVectorStore(faiss_index=faiss_index)

# 4. Retrieve the ingestion cache from your helper class
ingest_cache = SmartIngestionCache().get_cache()

# 5. Build a new pipeline that reuses transformations along with the document store, vector store, and cache
pipeline_with_store = IngestionPipeline(
    transformations=base_pipeline.transformations,  # reuse transformations
    docstore=SimpleDocumentStore(),
    vector_store=vector_store,
    cache=ingest_cache,
)

# 6. Load the persisted pipeline state if it exists
if os.path.exists(PERSIST_DIR):
    pipeline_with_store.load(PERSIST_DIR)
    print("Loaded pipeline state from persist directory.")
else:
    print("No persisted pipeline state found; processing documents.")

# 7. Run the pipeline (this will reuse cached transformations if available)
nodes = pipeline_with_store.run(documents=documents)

# 8. Build the index and retriever as before
index = VectorStoreIndex(nodes, vector_store=vector_store)
retriever = index.as_retriever(search_kwargs={"k": 7})

# 9. Persist the pipeline's cache and docstore state for future runs
pipeline_with_store.persist(PERSIST_DIR)
print("Pipeline state persisted.")

In [ ]:
# %%
print("Total number of vectors in FAISS index:", faiss_index.ntotal)
print("Number of nodes:", len(nodes))
print("Number of documents:", len(documents))

In [ ]:
print(nodes[6].metadata)

In [ ]:
%pip install tavily-python

In [ ]:
import asyncio
from tavily import AsyncTavilyClient
from llama_index.core.workflow import Context


async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient(api_key=TAVILY_API_KEY)
    return str(await client.search(query))


# async def search_faiss(query: str) -> str:
#     """
#     Useful for searchin the FAISS index for best description of the given query.
#     """
#     # Create a query engine from the index. In the newer API, use as_query_engine().
#     query_engine = index.as_query_engine(response_mode="detailed")
#     # Run the query in a thread-safe manner.
#     result = await asyncio.to_thread(query_engine.query, query)
#     return str(result)

async def search_faiss(query: str) -> str:
    """
    Useful for searching the FAISS index database using the retriever for the given query and returns a compact response.
    """
    # Run the retrieval in a thread to keep it asynchronous.
    results = await asyncio.to_thread(retriever.retrieve, query)
    # Combine the results into a single string, you can format it as needed.
    combined_results = "\n".join([str(node) for node in results])
    return combined_results


async def record_notes(ctx: Context, notes: str, notes_title: str) -> str:
    """Useful for recording notes on a given topic. Your input should be notes with a title to save the notes under."""
    current_state = await ctx.get("state")
    if "research_notes" not in current_state:
        current_state["research_notes"] = {}
    current_state["research_notes"][notes_title] = notes
    await ctx.set("state", current_state)
    return "Notes recorded."


async def write_report(ctx: Context, report_content: str) -> str:
    """Useful for writing a report on a given topic. Your input should be a markdown formatted report."""
    current_state = await ctx.get("state")
    current_state["report_content"] = report_content
    await ctx.set("state", current_state)
    return "Report written."


async def review_report(ctx: Context, review: str) -> str:
    """Useful for reviewing a report and providing feedback. Your input should be a review of the report."""
    current_state = await ctx.get("state")
    current_state["review"] = review
    await ctx.set("state", current_state)
    return "Report reviewed."

In [ ]:
# import asyncio
# from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow, AgentOutput, ToolCall, ToolCallResult, AgentStream
#
# # Create a test agent that uses only the FAISS search tool.
# test_agent = FunctionAgent(
#     name="TestAgent",
#     description="Agent for testing the FAISS search tool. When given a query, use the FAISS tool to retrieve information from the index.",
#     system_prompt=(
#         "You are TestAgent. When given a query, use the FAISS search tool to look up relevant information from the index."
#     ),
#     llm=Settings.llm,
#     tools=[search_faiss],
# )
#
# # Set up an agent workflow with just this one test agent.
# test_workflow = AgentWorkflow(
#     agents=[test_agent],
#     root_agent=test_agent.name,
#     initial_state={},
# )
#
# # Define an async function to trigger the agent and print out events.
# async def test_faiss_tool():
#     handler = test_workflow.run(
#         user_msg="Search the FAISS index for best description of AI implementation in life science industry."
#     )
#
#     async for event in handler.stream_events():
#         if hasattr(event, "current_agent_name"):
#             print(f"\n{'='*50}\nAgent: {event.current_agent_name}\n{'='*50}\n")
#         if isinstance(event, AgentOutput) and event.response.content:
#             print("Output:", event.response.content)
#         if isinstance(event, ToolCall):
#             print("Tool Call:", event.tool_name, event.tool_kwargs)
#         if isinstance(event, ToolCallResult):
#             print("Tool Call Result:", event.tool_name, event.tool_output)
#
# # Run the test.
# await test_faiss_tool()


In [ ]:
from llama_index.core.agent.workflow import FunctionAgent, ReActAgent

research_agent = FunctionAgent(
    name="ResearchAgent",
    description="Useful for searching the web and FAISS database for information on a given topic and recording notes on the topic.",
    system_prompt=(
    "You are ResearchAgent. Your task is to search both the web and FAISS index for comprehensive, "
    "detailed information on a given topic. Ensure that your search results include multiple aspects such as "
    "historical background, technical components, challenges, and future directions. Only after gathering sufficient "
    "detailed notes, hand off control to WriteAgent for drafting a full report."
    ),
    llm=Settings.llm,
    tools=[search_web, search_faiss, record_notes],
    can_handoff_to=["WriteAgent"],
)

write_agent = FunctionAgent(
    name="WriteAgent",
    description="Useful for writing a report on a given topic.",
    system_prompt=(
        "You are the WriteAgent that can write a report on a given topic. "
        "Your report should be in a markdown format. The content should be grounded in the research notes. "
        "Once the report is written, you should get feedback at least once from the ReviewAgent."
    ),
    llm=Settings.llm,
    tools=[write_report],
    can_handoff_to=["ReviewAgent", "ResearchAgent"],
)

review_agent = FunctionAgent(
    name="ReviewAgent",
    description="Useful for reviewing a report and providing feedback.",
    system_prompt=(
        "You are the ReviewAgent that can review the write report and provide feedback. "
        "Your review should either approve the current report or request changes for the WriteAgent to implement. "
        "If you have feedback that requires changes, you should hand off control to the WriteAgent to implement the changes after submitting the review."
    ),
    llm=Settings.llm,
    tools=[review_report],
    can_handoff_to=["WriteAgent"],
)

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

agent_workflow = AgentWorkflow(
    agents=[research_agent, write_agent, review_agent],
    root_agent=research_agent.name,
    initial_state={
        "research_notes": {},
        "report_content": "Not written yet.",
        "review": "Review required.",
    },
)

In [ ]:
import datetime
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

handler = agent_workflow.run(
    user_msg=(
        "Write me a report on the types of RAG. "
        "Briefly describe the sources that you used"
        "Shortly tell about advantages and disadvantages of all types of RAG."
    )
)

current_agent = None
current_tool_calls = ""
async for event in handler.stream_events():
    # print(f"\n🕒 {datetime.datetime.now()} | Event Type: {type(event).__name__}")

    if hasattr(event, "current_agent_name") and event.current_agent_name != current_agent:
        current_agent = event.current_agent_name
        print(f"\n🤖 Switched to Agent: {current_agent}")

    if isinstance(event, AgentInput):
        print("📥 Input Received:", event.input)

    if isinstance(event, AgentOutput):
        print("📤 Agent Output:", event.response.content)
        if event.tool_calls:
            print("🛠️ Tools Planned:", [call.tool_name for call in event.tool_calls])

    if isinstance(event, ToolCall):
        print(f"🔨 Tool Call - {event.tool_name}")
        print("   ↪ Arguments:", event.tool_kwargs)

    if isinstance(event, ToolCallResult):
        print(f"✅ Tool Result - {event.tool_name}")
        print("   ↪ Output:", event.tool_output)

    if hasattr(event, "state"):
        print("📊 State Snapshot:", event.state)


In [ ]:
state = await handler.ctx.get("state")
print(state["report_content"])